# Bai–Perron Multiple Structural-Break Test

For the Bai–Perron multiple structural-break test, Python does not have a single built-in `statsmodels` function equivalent to `het_arch()` for ARCH-LM. A practical approach is to use the **ruptures** library for multiple change-point detection.

Below is a reusable implementation that detects multiple breaks and makes a decision based on whether the detected breaks improve the segmentation significantly.


## 1. Install the required library


In [ ]:
# Run this cell once to install ruptures (if not already installed)
# !pip install ruptures

## 2. Python implementation


In [ ]:
import numpy as np
import pandas as pd
import ruptures as rpt


def bai_perron_test(
    series,
    significance=0.05,
    model="l2",
    penalty=None,
    min_size=20,
    name="Time Series"
):
    """
    Multiple structural-break detection inspired by the
    Bai-Perron framework.

    Parameters
    ----------
    series : array-like
        Time-series data.

    significance : float
        Significance level used for interpretation.

    model : str
        Cost model used by ruptures.
        "l2" is suitable for changes in the mean.

    penalty : float or None
        Penalty controlling the number of detected breaks.
        If None, BIC-type penalty is used.

    min_size : int
        Minimum number of observations in each regime.

    name : str
        Name of the time series.
    """

    # Convert to NumPy array
    y = np.asarray(series, dtype=float)

    # Remove missing values
    y = y[np.isfinite(y)]

    if len(y) < 2 * min_size:
        raise ValueError(
            "Series is too short for the selected min_size."
        )

    # Reshape for ruptures
    signal = y.reshape(-1, 1)

    # BIC-style penalty if not supplied
    if penalty is None:
        penalty = 3 * np.log(len(y))

    # Create change-point detection model
    algo = rpt.Pelt(
        model=model,
        min_size=min_size
    )

    # Fit model
    algo.fit(signal)

    # Detect breakpoints
    breakpoints = algo.predict(
        pen=penalty
    )

    # Last point is the end of the series,
    # not an actual structural break
    break_dates = [
        bp for bp in breakpoints
        if bp < len(y)
    ]

    # Create regimes
    boundaries = [0] + break_dates + [len(y)]

    regimes = []

    for i in range(len(boundaries) - 1):

        start = boundaries[i]
        end = boundaries[i + 1]

        regime = y[start:end]

        regimes.append({
            "Regime": i + 1,
            "Start": start,
            "End": end - 1,
            "Observations": len(regime),
            "Mean": np.mean(regime),
            "Std": np.std(regime, ddof=1)
        })

    regimes_df = pd.DataFrame(regimes)

    # Decision
    if len(break_dates) > 0:

        decision = "Reject H0"

        conclusion = (
            f"{len(break_dates)} structural break(s) "
            "detected."
        )

    else:

        decision = "Fail to Reject H0"

        conclusion = (
            "No structural breaks detected "
            "under the selected penalty."
        )

    # Display
    print("=" * 75)
    print("BAI-PERRON MULTIPLE STRUCTURAL BREAK ANALYSIS")
    print("=" * 75)

    print(f"Series              : {name}")
    print(f"Observations        : {len(y)}")
    print(f"Minimum regime size : {min_size}")
    print(f"Penalty             : {penalty:.4f}")
    print(f"Significance level  : {significance}")

    print("\nHypotheses:")
    print("H0: No structural breaks.")
    print("H1: One or more structural breaks exist.")

    print("\nDetected Breakpoints:")

    if len(break_dates) == 0:
        print("None")
    else:
        for bp in break_dates:
            print(f"Break at observation: {bp}")

    print("\nDecision:")
    print(decision)

    print("\nConclusion:")
    print(conclusion)

    print("\nDetected Regimes:")
    print(regimes_df.to_string(index=False))

    print("=" * 75)

    return {
        "breakpoints": break_dates,
        "number_of_breaks": len(break_dates),
        "regimes": regimes_df,
        "decision": decision,
        "conclusion": conclusion
    }

## 3. Test a series with structural breaks

Let's create an artificial series with three different regimes:


In [ ]:
np.random.seed(42)

regime_1 = np.random.normal(
    loc=100,
    scale=2,
    size=150
)

regime_2 = np.random.normal(
    loc=120,
    scale=2,
    size=150
)

regime_3 = np.random.normal(
    loc=90,
    scale=2,
    size=150
)

data = np.concatenate([
    regime_1,
    regime_2,
    regime_3
])

print(f"Total observations: {len(data)}")
print(f"True break points should be near: 150 and 300")

Run the test:


In [ ]:
result = bai_perron_test(
    data,
    min_size=30,
    name="Simulated Series"
)

You should detect breakpoints close to:

```
Observation ≈ 150
Observation ≈ 300
```

The exact locations can vary because the data contain random noise.


### Optional: Visualize the series and detected breaks


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(data, color="steelblue", linewidth=1, label="Series")

# Plot true regime means for reference
ax.axhline(100, color="gray", linestyle="--", alpha=0.5, xmax=150/450)
ax.axhline(120, color="gray", linestyle="--", alpha=0.5, xmin=150/450, xmax=300/450)
ax.axhline(90, color="gray", linestyle="--", alpha=0.5, xmin=300/450)

# Detected breaks
for bp in result["breakpoints"]:
    ax.axvline(bp, color="crimson", linestyle="-", linewidth=1.5, label="Detected break" if bp == result["breakpoints"][0] else None)

ax.set_title("Simulated Series with Detected Structural Breaks")
ax.set_xlabel("Observation")
ax.set_ylabel("Value")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Decision

Conceptually:

```
H0:
No structural breaks

H1:
One or more structural breaks
```

**If breaks are detected:**

```
Structural breaks detected
        ↓
Reject H0
        ↓
Time series has different regimes
```

**If no breaks are detected:**

```
No structural breaks detected
        ↓
Fail to Reject H0
        ↓
No evidence of breaks
under the chosen specification
```

### Important caveat

With **ruptures**, the decision is **not** a classical p-value-based Bai–Perron hypothesis test. The penalty controls how many breaks are detected. Therefore, don't interpret "Reject H0" above as equivalent to a formal Bai–Perron p-value.

For a strict econometric Bai–Perron implementation, the break-selection procedure should use the Bai–Perron statistics/criteria such as **supF tests**, sequential tests, **BIC/Schwarz information criterion**, or **LWZ**, depending on the specification.


## 5. Applying it to stock prices

For example (replace with your own CSV path):

```python
df = pd.read_csv("stock_data.csv")

prices = df["Close"].dropna()

result = bai_perron_test(
    prices,
    min_size=50,
    name="Stock Close Price"
)
```

However, for financial analysis, it is often more meaningful to test **returns**:

```python
returns = df["Close"].pct_change().dropna()

result = bai_perron_test(
    returns,
    min_size=50,
    name="Stock Returns"
)
```

You might get output similar to:

```
Detected Breakpoints:
Break at observation: 245
Break at observation: 512
Break at observation: 780

Decision:
Reject H0

Conclusion:
3 structural breaks detected.
```

This suggests the series can be divided into approximately:

```
Regime 1
0 ─────── 245

Regime 2
245 ───── 512

Regime 3
512 ───── 780

Regime 4
780 ───── End
```

You can then calculate statistics separately for each regime.


### Demo: Simulated returns-like series (optional)


In [ ]:
np.random.seed(123)

# Simulate returns with a volatility/mean shift
r1 = np.random.normal(0.0005, 0.01, 200)
r2 = np.random.normal(-0.001, 0.02, 200)   # different mean & higher vol
r3 = np.random.normal(0.001, 0.008, 150)

returns_sim = np.concatenate([r1, r2, r3])

result_ret = bai_perron_test(
    returns_sim,
    min_size=40,
    name="Simulated Returns"
)

---
**Note:** This notebook uses the `ruptures` library (PELT algorithm) as a practical approximation for multiple change-point detection inspired by the Bai–Perron framework. For formal econometric inference, consider specialized packages or implementations that report Bai–Perron test statistics.
